# S00 — Prompt Engineering Foundations Lab

## Why this notebook exists

Imagine you have just joined a customer conversation at **Northstar**, a fictional company.
Leadership has two questions: *What are customers saying?* and *What might be driving
churn?* Before proposing an application or an automated agent, you need to learn whether a
carefully designed prompt can produce a useful, evidence-based first answer.

This notebook introduces prompt engineering as a practical problem-solving skill. A prompt is
more than a question: it can specify a role, provide examples, constrain valid answers, supply
evidence, and define the structure of the response. You will see how those choices affect both
the quality of an answer and how confidently it can be used.

## What you will learn

By the end of the lab, you should be able to:

1. Explain the difference between a vague instruction and a prompt with examples and constraints.
2. Use a small labeled set to evaluate output instead of relying on whether it merely looks good.
3. Group feedback into useful themes while allowing uncertain cases to remain unclassified.
4. Ground a narrative in computed evidence and recognize misleading small-sample findings.
5. Turn analysis into a concise stakeholder recommendation with an explicit caveat and next step.

## How the lab fits together

You will move from raw data to a business recommendation in seven parts: inspect the data,
compare naive and example-guided classification, organize feedback by meaning, calculate churn
patterns, design a structured narrative prompt, optionally try that prompt with a live model, and
finally write the recommendation yourself. Most of the lab runs offline so you can focus on the
ideas and inspect every result. Run the cells in order, pause at the reflection prompts, and edit
the learner exercises rather than treating the notebook as a read-only demo.


## 🧩 Part 1 — Load the data

**Goal:** understand the evidence before asking a model to interpret it. Prompt quality cannot
compensate for misunderstood columns, missing context, or unsuitable data. In this part you will
load both datasets, check their size, and preview their structure.

Two files back this lab: `data/user_comments.csv` (100 rows, ~28 unique phrasings — the
same comments the handout's Few-Shot and Semantic Filter examples are drawn from) and
`data/churn_modelling.csv` (10,000 synthetic-style bank customer records — the dataset the
handout's Chain-of-Thought / Meta Prompt example is built on).

As you inspect the output, ask: What does one row represent? Which columns contain direct
customer language, which contain measured attributes, and what important context is absent?


In [ ]:
# %pip install pandas
import pandas as pd

comments = pd.read_csv("data/user_comments.csv")
churn = pd.read_csv("data/churn_modelling.csv")

print(f"comments: {len(comments)} rows, {comments['comment'].nunique()} unique phrasings")
print(f"churn:    {len(churn)} rows, {len(churn.columns)} columns")
comments.head()


In [ ]:
churn.head()


## 🧩 Part 2 — Naive vs. few-shot sentiment classification

**Goal:** see why examples are part of the specification, not decoration. A broad instruction
such as 'classify sentiment' leaves boundary cases undefined. Few-shot prompting supplies worked
examples that demonstrate how labels should be applied to cases such as negation and hedging.

Two **deterministic, offline stand-ins** simulate what a model would plausibly return
under two different prompting approaches on the same comments:

- `naive_classify` — a keyword-counting rule with **no negation handling**. This stands in
  for a zero-shot prompt that just says "classify as Positive/Neutral/Negative" with no
  worked examples.
- `fewshot_classify` — a lookup informed by the kind of boundary-case examples shown in the
  handout's §3.4 (negation, hedging, cancellation). This stands in for a prompt that
  includes those worked examples.

Neither function calls a real model — see the lab README's **Scope / limitations** for why,
and the quiz's Q10/Q11 for why even the "good" result here isn't yet a defensible claim on
its own.

`GOLD_LABELS` is this session's own hand-labeled ground truth for the 28 unique comment
phrasings in `data/user_comments.csv`. The comparison is an analogy for prompt behavior, not a
benchmark of a real language model. Focus on *which* errors change and why, not only the final
accuracy percentage.


In [ ]:
GOLD_LABELS = {
    "Product is okay, nothing special.": "Neutral",
    "Fantastic purchase, will buy again!": "Positive",
    "Highly recommend to everyone.": "Positive",
    "Customer service was unhelpful.": "Negative",
    "Quality is acceptable.": "Neutral",
    "Shipping took forever.": "Negative",
    "Would not recommend to anyone.": "Negative",
    "Absolutely love this product!": "Positive",
    "Meh, it’s alright.": "Neutral",
    "Neither good nor bad.": "Neutral",
    "Service was okay.": "Neutral",
    "Average experience overall.": "Neutral",
    "It’s decent for the price.": "Neutral",
    "Top-notch item, worth the price.": "Positive",
    "Excellent service and amazing quality.": "Positive",
    "Very disappointed with the quality.": "Negative",
    "Great quality and fast delivery.": "Positive",
    "Defective item received.": "Negative",
    "Product broke after one use.": "Negative",
    "Not as described, waste of money.": "Negative",
    "It does the job, but not great.": "Neutral",
    "Poor quality and overpriced.": "Negative",
    "Impressive packaging and great value.": "Positive",
    "Product works fine, but delivery was slow.": "Neutral",
    "Product works perfectly.": "Positive",
    "Packaging was damaged upon arrival.": "Negative",
    "Exceeded my expectations.": "Positive",
    "Not bad, but could be better.": "Neutral",
}

assert set(GOLD_LABELS) <= set(comments["comment"].unique()), (
    "A gold-set phrasing doesn't match the CSV — check for a smart-quote mismatch."
)
print(f"{len(GOLD_LABELS)} gold-labeled phrasings")


In [ ]:
POSITIVE_WORDS = {
    "fantastic", "recommend", "love", "excellent", "amazing", "great",
    "impressive", "top-notch", "exceeded", "perfectly", "worth", "good",
}
NEGATIVE_WORDS = {
    "unhelpful", "forever", "disappointed", "defective", "broke", "poor",
    "damaged", "waste", "overpriced", "slow", "bad",
}


def naive_classify(comment: str) -> str:
    """Zero-shot stand-in: counts keyword hits, blind to negation."""
    text = comment.lower()
    pos_hits = sum(1 for w in POSITIVE_WORDS if w in text)
    neg_hits = sum(1 for w in NEGATIVE_WORDS if w in text)
    if pos_hits > neg_hits:
        return "Positive"
    if neg_hits > pos_hits:
        return "Negative"
    return "Neutral"


# A deliberately imperfect lookup: correct on every boundary case the naive
# rule gets wrong, except one genuinely ambiguous comment left as an honest
# residual disagreement -- no prompting approach is perfect, mock or real.
_FEWSHOT_OVERRIDE = {"Product works fine, but delivery was slow.": "Negative"}


def fewshot_classify(comment: str) -> str:
    """Few-shot stand-in: informed by worked negation/hedge examples."""
    if comment in _FEWSHOT_OVERRIDE:
        return _FEWSHOT_OVERRIDE[comment]
    return GOLD_LABELS.get(comment, naive_classify(comment))


In [ ]:
rows = []
for sentence, gold in GOLD_LABELS.items():
    rows.append({
        "comment": sentence,
        "gold": gold,
        "naive": naive_classify(sentence),
        "few_shot": fewshot_classify(sentence),
    })

results = pd.DataFrame(rows)
naive_acc = (results["gold"] == results["naive"]).mean()
fewshot_acc = (results["gold"] == results["few_shot"]).mean()
print(f"naive accuracy:    {naive_acc:.0%}")
print(f"few-shot accuracy: {fewshot_acc:.0%}")

print("\nnaive mismatches:")
results[results["gold"] != results["naive"]]


In [ ]:
print("few-shot mismatches:")
results[results["gold"] != results["few_shot"]]


> [!NOTE]
> Both mismatch tables matter more than the two accuracy numbers above them. Read each
> mismatch and confirm it's the negation/hedging failure the handout describes (§3.4), not
> a random miss — that's the difference between "the pattern fixed a real, predictable
> problem" and "the number moved for an unclear reason."

### Mini exercise

Write one comment of your own, designed to defeat `naive_classify`'s keyword rule (use a
negation or a hedge the word lists can't see through). Confirm `fewshot_classify` doesn't
have an answer for it either unless you add it to `GOLD_LABELS` first — that's the point:
a lookup only "knows" what it's been shown, same as a few-shot prompt only generalizes as
far as its examples actually cover.


In [ ]:
my_comment = "Replace this with your own tricky comment."
print(f"naive:    {naive_classify(my_comment)}")
print(f"few-shot: {fewshot_classify(my_comment)}")


## 🧩 Part 3 — Semantic Filter: theme the negative feedback

**Goal:** turn individual comments into categories a team can act on without pretending every
comment is specific enough to classify. A semantic filter uses meaning and business definitions,
rather than word matching alone, to decide whether an item belongs in an allowed category.

Northstar's support lead doesn't want 100 loose comments — they want **themes**: Delivery,
Pricing, Product Quality, Customer Support. The grouping below is by *meaning*, not shared
keywords — notice that "Shipping took forever" and "Packaging was damaged upon arrival"
land in different themes despite both being about the physical delivery, and that one
comment below deliberately doesn't fit any of the four themes at all. This is an important prompt
pattern: define the allowed labels, describe their meaning, and include a safe fallback such as
`None` when the evidence does not justify a label.


In [ ]:
NEGATIVE_THEMES = {
    "Customer service was unhelpful.": "Customer Support",
    "Shipping took forever.": "Delivery",
    "Very disappointed with the quality.": "Product Quality",
    "Defective item received.": "Product Quality",
    "Product broke after one use.": "Product Quality",
    "Not as described, waste of money.": "Pricing",
    "Poor quality and overpriced.": "Product Quality + Pricing",
    "Packaging was damaged upon arrival.": "Delivery",
    "Would not recommend to anyone.": None,  # too general to theme -- flagged, not forced
}

for sentence, theme in NEGATIVE_THEMES.items():
    label = theme if theme else "UNTHEMED (too general -- needs a human read, not a forced bucket)"
    print(f"- {sentence!r}\n    -> {label}")


> [!IMPORTANT]
> Forcing "Would not recommend to anyone." into one of the four themes would be worse than
> leaving it unthemed — it names no specific aspect of the experience, so any theme you
> picked would be a guess the support lead can't act on. A real Semantic Filter prompt
> should give the model explicit permission to say "none of the above."


## 🧩 Part 4 — Churn EDA (the numbers behind the Chain-of-Thought example)

**Goal:** calculate the evidence that a later prompt will summarize. Exploratory data analysis
(EDA) helps you find patterns worth discussing and exposes details—such as group size—that can
be lost when only a polished narrative is shown.

This is the computation behind the handout's §3.2 and §4 examples and the original
Julius.ai-style prompt this session is built from: *"Analyze this telecom churn dataset to
identify key factors contributing to customer churn... visualize churn distribution across
contract types and tenure... summarize actionable insights."* Here, the numbers are real
`pandas` arithmetic on the provided CSV -- nothing below is simulated. The grouped churn rates
show associations in this dataset; they do **not** prove that geography, activity, product count,
or tenure causes churn. Keep that distinction in mind when writing recommendations.

Before moving on, identify one strong pattern, one surprising pattern, and one question you would
ask before advising Northstar to act.


In [ ]:
overall_churn = churn["Exited"].mean()
by_geography = churn.groupby("Geography")["Exited"].mean().sort_values(ascending=False)
by_active = churn.groupby("IsActiveMember")["Exited"].mean()
by_products = churn.groupby("NumOfProducts")["Exited"].mean().sort_values(ascending=False)

print(f"Overall churn rate: {overall_churn:.1%}\n")
print("Churn rate by geography:")
print(by_geography.apply(lambda x: f'{x:.1%}'))
print("\nChurn rate by active-member status (0 = inactive, 1 = active):")
print(by_active.apply(lambda x: f'{x:.1%}'))
print("\nChurn rate by number of products held:")
print(by_products.apply(lambda x: f'{x:.1%}'))


In [ ]:
tenure_bucket = pd.cut(
    churn["Tenure"], bins=[-1, 2, 5, 8, 10], labels=["0-2 yrs", "3-5 yrs", "6-8 yrs", "9-10 yrs"]
)
by_tenure = churn.groupby(tenure_bucket)["Exited"].mean()
print("Churn rate by tenure bucket:")
by_tenure.apply(lambda x: f"{x:.1%}")


### Learner modification

Add one more cut of the data on your own — e.g. a credit-score band using `pd.cut`, the
same way the tenure bucket above was built. Does the new cut change which single finding
you'd lead with in Part 7's stakeholder note, or does it just add detail to the same
headline?


In [ ]:
# Your own cut of the churn data -- e.g. by CreditScore band.
# credit_bucket = pd.cut(churn["CreditScore"], bins=[...], labels=[...])
# churn.groupby(credit_bucket)["Exited"].mean()


## 🧩 Part 5 — Draft the Meta Prompt + Role + Structure narrative

**Goal:** convert verified calculations into a prompt that produces a focused stakeholder
narrative. This combines three reusable prompt patterns: a **role** sets the perspective, supplied
**data points** ground the answer, and an explicit **output structure** makes the response easier
to review and compare.

This is the prompt you'd actually paste into a tool like Julius.ai once you have numbers
like the ones above. Fill in the blanks below using *your own* computed values from Part 4
— the f-string pulls the real numbers in automatically so your draft can't drift from what
the data actually says. Notice that an f-string fills in computed values, but it does not decide
whether those values are representative or decision-worthy. Evidence selection remains your job.


In [ ]:
narrative_prompt = f"""\
Act as a data storytelling expert. I have Northstar's quarterly churn numbers to present
to senior leadership. Convert these data points into a short narrative with three
sections: Context, Key Insight, Call-to-Action.

Data points:
- Overall churn rate is {overall_churn:.1%}
- {by_geography.index[0]}'s churn rate ({by_geography.iloc[0]:.1%}) is the highest of any geography
- Customers with {by_products.index[0]} product(s) churn at {by_products.iloc[0]:.1%}, the highest of any product-count group
"""

print(narrative_prompt)


> [!WARNING]
> Look at what the auto-filled prompt above just asserted: the highest-churning
> product-count group is "100%". Check Part 4's `by_products` output again -- that 100%
> comes from a 60-customer group, out of 10,000. It's literally true and still a
> misleading headline if it displaces the much larger, more decision-relevant finding
> (1-product customers churn at 27.7% vs. 7.6% for 2-product customers, across roughly
> 9,700 customers). A real Meta Prompt should state the group size alongside the rate, or
> this kind of small-sample finding will win the "most dramatic number" contest every time
> and crowd out the finding that's actually safe to act on.


## 💻 Part 6 (optional) — Run a real prompt against a live model

**Goal:** observe the difference between constructing a prompt and sending it to a model. The
previous section produced ordinary text; this section optionally submits that text through an API
and prints the model's response. The model receives only the content in `narrative_prompt`—it does
not automatically know the dataframe, earlier notebook cells, or Northstar's unstated context.

Off by default. From the repo root: `uv sync --group live`, then copy `.env.example` to
`.env` and set `OPENROUTER_API_KEY` there — see this lab's `README.md` for the full setup
and the caution about never pasting real customer data here.

In [ ]:
import os

RUN_LIVE = False
LIVE_MODEL = "openrouter/auto"  # replace with a verified OpenRouter model ID

if RUN_LIVE:
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv()  # reads .env from the repo root (or this folder, if present)
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY before setting RUN_LIVE = True.")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    response = client.chat.completions.create(
        model=LIVE_MODEL,
        messages=[{"role": "user", "content": narrative_prompt}],
    )
    print(response.choices[0].message.content)
else:
    print("RUN_LIVE is False -- set it to True only after reading the README's caution note.")


## 🛠️ Part 7 — FDE decision exercise

**Goal:** practice the final step of forward-deployed work: translating technical evidence into a
decision-ready message. A useful stakeholder note is not a dump of every metric. It prioritizes
one defensible finding, states what the analysis cannot establish, and recommends a proportionate
next action.

Write the 3–5 sentence note you'd actually send Northstar's VP of Support, using your own
Part 4 numbers. Include the headline finding, one honest caveat about the data or method,
and a concrete next step. This is the same skill the quiz's Q13 and the interview bank's
Q13 test — practice it here first, then compare your note against the quiz's example
answer.


_Write your stakeholder note here._


## 🗺️ Where this goes next

A prompt-driven read like this one is enough to earn the next meeting. It is **not**
enough to automate an unreviewed business action — see the handout's §5 and the quiz's
Q14. If Northstar asks for that next, the missing mechanics are in **S01: Verified Agent
Loops** (who's allowed to act, bounded retries, when to escalate) and **S02: Structured
Outputs and Model Selection** (a typed, validated decision contract the automation can
safely consume).
